# Hartland Quay (Devon) — numbers for the text

Every Hartland number the paper quotes that the fitting and figure notebooks do not already print,
for the LDI member the figure shows (`MEMBER`, as in `05d`) and for the Surfe fit of `05c`.
Nothing is fitted. Reads `data/processed/` (05a), `results/ensemble/member_NN.pt` (05b) and
`results/surfe.npz` (05c); writes `results/numbers_member_NN.json`.

1. bedding misfit on the cliff and on the wave-cut platform;
2. the height of the restored sequence, and traces that restore to one depth;
3. shortening, from line length and from the restored traces;
4. interlimb angles at the hinges 05a found in the traces, in the traces and in the model;
5. where Surfe's interior extrema sit relative to the data and the outcrop.

## Setup

In [9]:
import json
from pathlib import Path

import numpy as np
import torch
import contourpy
from scipy.ndimage import gaussian_filter1d
from scipy.spatial import cKDTree

import curlew
from curlew.io import loadModel

import hartland_inputs

curlew.device = "cpu"
curlew.dtype = torch.float32
curlew.default_dim = 3

MEMBER = 19                 # the member Fig. 8 shows (05d)

RES_DIR = hartland_inputs.RESULTS_DIR
data = hartland_inputs.load_processed()
folds = np.load(hartland_inputs.PROCESSED_DIR / "hartland_processed.npz")   # 05a's fold sweep: folded traces, hinges
traces, trace_names = data["traces"], data["trace_names"]
hinge, u_ax, w_ax = data["hinge"], data["u_ax"], data["w_ax"]
z_beach = data["settings"]["z_beach"] - data["origin"][2]      # the platform's top, in the modelling frame
model = loadModel(RES_DIR / "ensemble" / f"member_{MEMBER:02d}.pt")
field = model.events[0].field
metrics = np.load(RES_DIR / "ensemble" / "metrics.npz")
numbers = {"member": MEMBER}
T = lambda a: torch.as_tensor(np.asarray(a, float), dtype=curlew.dtype)


def scalar(xyz):
    with torch.no_grad():
        return np.concatenate([np.asarray(model.predict(T(c), litho=False).scalar).ravel()
                               for c in np.array_split(np.asarray(xyz, float), max(1, len(xyz) // 50_000))])


print(f"member {MEMBER}: {len(traces)} fitted traces, {len(data['gp_fit'])} fitted bedding, platform top z = {z_beach:.1f} m")

member 19: 130 fitted traces, 147 fitted bedding, platform top z = -5.5 m


## 1. Bedding misfit, cliff and platform

The angle between each fitted bedding normal and the one the model predicts there, split by
05a's `on_beach_fit` (the wave-cut platform) and the rest (the cliff).

In [10]:
gp, gv, on_beach = data["gp_fit"], data["gv_fit"], data["on_beach_fit"]
with torch.no_grad():
    pred = field.predict_normals(T(gp)).numpy()
obs = gv / np.linalg.norm(gv, axis=1, keepdims=True)
pred = pred / np.linalg.norm(pred, axis=1, keepdims=True).clip(min=1e-12)
angle = np.degrees(np.arctan2(np.linalg.norm(np.cross(obs, pred), axis=1), np.clip((obs * pred).sum(1), -1, 1)))
numbers["bedding_deg_median"] = float(np.median(angle))
numbers["bedding_deg_median_cliff"] = float(np.median(angle[~on_beach]))
numbers["bedding_deg_median_platform"] = float(np.median(angle[on_beach]))
print(f"bedding misfit: all {numbers['bedding_deg_median']:.1f}° ({len(angle)}), "
      f"cliff {numbers['bedding_deg_median_cliff']:.1f}° ({(~on_beach).sum()}), "
      f"platform {numbers['bedding_deg_median_platform']:.1f}° ({on_beach.sum()})")

bedding misfit: all 1.5° (147), cliff 1.5° (53), platform 1.5° (94)


## 2. Restored sequence

Each fitted trace pulled back through the map: its mean restored depth places it in the sequence,
its standard deviation is its misfit. The sequence height is the range of the mean depths.

**Traces on one contact.** Two traces are taken as candidates for one contact when each is the
other's nearest trace in restored depth (mutual nearest neighbours) and their mean depths differ by
less than `PAIR_FACTOR` times the median misfit.

In [11]:
PAIR_FACTOR = 2.0

with torch.no_grad():
    restored = [field.inverse_map(T(t)).numpy() for t in traces]
depth = np.array([r[:, 2].mean() for r in restored])
depth_sd = np.array([r[:, 2].std(ddof=1) for r in restored])   # sample sd, as 05b (torch.std)
numbers["depth_sd_median_m"] = float(np.median(depth_sd))
numbers["sequence_height_m"] = float(np.ptp(depth))

order = np.argsort(depth)
d_sorted = depth[order]
gap = np.diff(d_sorted)
nearest = np.where(np.r_[np.inf, gap] < np.r_[gap, np.inf], -1, 1)    # -1: nearest is below, +1: above
threshold = PAIR_FACTOR * numbers["depth_sd_median_m"]
pairs = [(order[i], order[i + 1]) for i in range(len(gap))
         if nearest[i] == 1 and nearest[i + 1] == -1 and gap[i] < threshold]
numbers["pairs_threshold_m"] = float(threshold)
numbers["pairs_n"] = len(pairs)
print(f"restored-depth sd median {numbers['depth_sd_median_m']:.2f} m; sequence height {numbers['sequence_height_m']:.1f} m "
      f"({depth.min():+.1f} to {depth.max():+.1f} m)")
print(f"median gap between neighbouring traces in restored depth {np.median(gap):.2f} m "
      f"(the pair threshold is {threshold:.2f} m)")
print(f"{len(pairs)} mutual-nearest pairs within {threshold:.2f} m:",
      ", ".join(f"{trace_names[a]}–{trace_names[b]} {abs(depth[a] - depth[b]):.2f}" for a, b in pairs))

restored-depth sd median 0.59 m; sequence height 100.2 m (-9.1 to +91.1 m)
median gap between neighbouring traces in restored depth 0.34 m (the pair threshold is 1.18 m)
39 mutual-nearest pairs within 1.18 m: H92–H90 0.04, H89–H94 0.23, H95–H88 0.11, H97–H96 0.24, H83–H33 0.05, H98–H87 0.03, H78–H76 0.15, H79–H86 0.00, H77–H106 0.16, H99–H108 0.03, H74–H69 0.00, H73–H115 0.01, H120–H103 0.11, H100–H37 0.02, H101–H102 0.02, H110–H36 0.00, H109–H35 0.01, H121–H34 0.14, H72–H111 0.03, H66–H18 0.14, H125–H64 0.37, H126–H4 0.13, H55–H54 0.58, H128–H5 0.19, H16–H122 0.13, H129–H26 0.14, H123–H39 0.00, H38–H57 0.23, H61–H15 0.06, H58–H6 0.59, H59–H43 0.83, H131–H44 0.02, H14–H32 0.04, H13–H45 0.40, H41–H3 0.01, H47–H46 0.11, H42–H31 0.41, H24–H27 0.44, H11–H50 0.60


## 3. Shortening

For the traces 05a found folded (`fold_trace_names`) that are in the fit, two estimates of the
shortening across the hinge, each bed by bed, as a bed-length balance along a cliff is done, and summarised
by the median and range over the beds:

- **line length**, from the data alone: the trace's length in the fold profile against its present
  width across the hinge (along `u_ax`), as a bed-length balance of the cliff would measure it;
- **restored**, from the model: the trace's restored width across the hinge against its present width.
  The map moves material only in planes normal to the hinge, so in the restored frame the hinge keeps
  its direction and the width is measured along the horizontal normal to it.

In [12]:
DS = 0.5                  # m, resampling step along a profile polyline (as 05a)

def resample(p, ds=DS):
    s = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(p, axis=0), axis=1))]
    keep = np.r_[True, np.diff(s) > 1e-9]
    p, s = p[keep], s[keep]
    si = np.arange(0.0, s[-1], ds)
    return np.stack([np.interp(si, s, p[:, c]) for c in range(p.shape[1])], axis=1)


r_ax = np.cross(hinge, [0.0, 0.0, 1.0])
r_ax /= np.linalg.norm(r_ax)
folded = [k for k, n in enumerate(trace_names) if n in set(map(str, folds["fold_trace_names"]))]
rows = []
for k in folded:
    q = traces[k] @ np.c_[u_ax, w_ax]
    width = float(np.ptp(q[:, 0]))
    length = float(np.linalg.norm(np.diff(resample(q), axis=0), axis=1).sum())
    restored_width = float(np.ptp(restored[k] @ r_ax))
    rows.append((trace_names[k], width, length, restored_width))
    print(f"{trace_names[k]:5s} width {width:6.1f} m, profile length {length:6.1f} m, restored width {restored_width:6.1f} m")
W, L, W0 = (np.array([r[i] for r in rows]) for i in (1, 2, 3))
s_line = 100 * (1 - W / L)                  # bed by bed: present width against the bed's length in the profile
s_rest = 100 * (1 - W / W0)                 # bed by bed: present width against its restored width
numbers["shortening_line_length_pct"] = [float(v) for v in s_line]
numbers["shortening_restored_pct"] = [float(v) for v in s_rest]
numbers["shortening_traces"] = [r[0] for r in rows]
for (name, *_), a, b in zip(rows, s_line, s_rest):
    print(f"{name:5s} shortening {a:3.0f}% by line length, {b:3.0f}% from the restored trace")
print(f"{len(rows)} folded beds in the fit, bed by bed: by line length median {np.median(s_line):.0f}% "
      f"({s_line.min():.0f}–{s_line.max():.0f}%), from the restored traces median {np.median(s_rest):.0f}% "
      f"({s_rest.min():.0f}–{s_rest.max():.0f}%)")

H4    width   27.2 m, profile length   68.6 m, restored width   36.3 m
H8    width   58.5 m, profile length   99.3 m, restored width   87.9 m
H13   width   19.3 m, profile length   37.0 m, restored width   33.1 m
H24   width   48.6 m, profile length  100.8 m, restored width   72.9 m
H26   width   29.0 m, profile length   73.2 m, restored width   47.2 m
H27   width   32.1 m, profile length   57.9 m, restored width   47.8 m
H28   width   34.7 m, profile length   81.9 m, restored width   58.8 m
H29   width   19.3 m, profile length   45.9 m, restored width   42.2 m
H4    shortening  60% by line length,  25% from the restored trace
H8    shortening  41% by line length,  33% from the restored trace
H13   shortening  48% by line length,  42% from the restored trace
H24   shortening  52% by line length,  33% from the restored trace
H26   shortening  60% by line length,  39% from the restored trace
H27   shortening  45% by line length,  33% from the restored trace
H28   shortening  58% by line 

## 4. Interlimb angles

At each hinge 05a found in the traces (`hinge_trace`, `hinge_uw`), the interlimb angle is the
angle between the limbs at the inflection points on either side: 180° minus the turn of the
tangent between them. The tangent angle is smoothed over `SMOOTH`, as in 05a. Walking out from the
hinge, the inflection point is where the tangent stops turning (the curvature changes sign), or
`LIMB_MAX` of arc if it has not by then.

The same measure is taken on the trace itself (the data; only the stretch of it that crosses the folds,
as 05a cut it, since stretches along the hinge collapse to jitter in the profile) and on the model's contour through the
same point, in the fold profile through it (the model). The model's hinge is the curvature peak
within `HINGE_SEARCH` of the data hinge; where that peak is at the edge of the search, the model has
no hinge there and the hinge is left out of the model's summary.

In [13]:
SMOOTH = 2.0                # m, smoothing of the tangent angle (as 05a)
LIMB_MAX = 30.0             # m of arc, furthest an inflection point is looked for
HINGE_SEARCH = 5.0          # m of arc, where the model's hinge is looked for around the data hinge
SEC_HALF, SEC_STEP = 45.0, 0.5   # m, half-width and step of each model profile section


MAX_ALONG_HINGE, BRIDGE = 0.5, 3.0   # as 05a: stretches of a trace running along the hinge are not fold profile


def profile_run(t, centre):
    """The stretch of trace t that crosses the folds (05a's profile run) and contains the hinge at `centre`,
    in (u, w). Stretches running along the hinge would collapse to jitter in the profile, so they are cut,
    and gaps shorter than BRIDGE are kept, as in 05a."""
    r = resample(t)
    tan = np.gradient(gaussian_filter1d(r, SMOOTH / DS, axis=0, mode="nearest"), axis=0)
    tan /= np.linalg.norm(tan, axis=1, keepdims=True).clip(min=1e-12)
    good = np.abs(tan @ hinge) < MAX_ALONG_HINGE
    gaps = np.flatnonzero(np.diff(np.r_[1, good.astype(int), 1]))
    for a, b in zip(gaps[::2], gaps[1::2]):
        if 0 < a and b < len(good) and (b - a) * DS < BRIDGE:
            good[a:b] = True
    q = r @ basis
    edges = np.flatnonzero(np.diff(np.r_[0, good.astype(int), 0]))
    runs = [q[a:b] for a, b in zip(edges[::2], edges[1::2]) if b - a > 1]
    return min(runs, key=lambda run: np.linalg.norm(run - centre, axis=1).min())


def interlimb(q, centre, search):
    """Interlimb angle (deg) of the profile polyline q (n, 2) at its hinge nearest `centre`."""
    q = resample(q)
    d = np.gradient(q, axis=0)
    theta = gaussian_filter1d(np.unwrap(np.arctan2(d[:, 1], d[:, 0])), SMOOTH / DS, mode="nearest")
    kappa = np.gradient(theta) / DS
    i0 = int(np.argmin(np.linalg.norm(q - centre, axis=1)))
    w = int(round(search / DS))
    lo, hi = max(i0 - w, 1), min(i0 + w + 1, len(q) - 1)
    h = lo + int(np.argmax(np.abs(kappa[lo:hi])))
    sense = np.sign(kappa[h])
    n = int(round(LIMB_MAX / DS))
    ends = []
    for step in (-1, 1):
        j = h
        while 0 < j < len(q) - 1 and abs(j - h) < n and np.sign(kappa[j + step]) == sense:
            j += step
        ends.append(j)
    return 180.0 - np.degrees(abs(theta[ends[1]] - theta[ends[0]])), q[h]


all_names, all_traces = data["all_trace_names"], data["all_traces"]
basis = np.c_[u_ax, w_ax]
uu = np.arange(-SEC_HALF, SEC_HALF + 1e-9, SEC_STEP)
rows = []
for name, uw in zip(map(str, folds["hinge_trace"]), folds["hinge_uw"]):
    t = all_traces[all_names.index(name)]
    q = t @ basis
    p = t[int(np.argmin(np.linalg.norm(q - uw, axis=1)))]          # the hinge, in 3-D
    data_angle, _ = interlimb(profile_run(t, uw), uw, HINGE_SEARCH)
    # The model's profile through p: a grid in (u, w) around the hinge, at p's position along the hinge.
    U, Wg = np.meshgrid(uw[0] + uu, uw[1] + uu)
    xyz = (p @ hinge) * hinge + U.ravel()[:, None] * u_ax + Wg.ravel()[:, None] * w_ax
    lines = contourpy.contour_generator(uw[0] + uu, uw[1] + uu, scalar(xyz).reshape(U.shape),
                                        line_type="Separate").lines(float(scalar(p[None])[0]))
    line = min(lines, key=lambda l: np.linalg.norm(l - uw, axis=1).min())
    model_angle, model_hinge = interlimb(line, uw, HINGE_SEARCH)
    off = float(np.linalg.norm(model_hinge - uw))
    found = off < HINGE_SEARCH - DS          # a model hinge at the edge of the search is not a hinge found
    rows.append((name, data_angle, model_angle, off, found))
    print(f"{name:5s} at u={uw[0]:6.1f}, w={uw[1]:5.1f}: interlimb {data_angle:5.0f}° in the trace, "
          f"{model_angle:5.0f}° in the model (its hinge {off:.1f} m away{'' if found else ', at the search edge: left out'})")
keep = np.array([r[4] for r in rows])
data_ang, model_ang = np.array([r[1] for r in rows]), np.array([r[2] for r in rows])[keep]
numbers["interlimb_data_deg"] = [float(a) for a in data_ang]
numbers["interlimb_model_deg"] = [float(a) for a in model_ang]
print(f"interlimb, traces at {len(rows)} hinges: median {np.median(data_ang):.0f}° "
      f"(25–75th {np.percentile(data_ang, 25):.0f}–{np.percentile(data_ang, 75):.0f}°), "
      f"model at the {keep.sum()} hinges it reproduces: median {np.median(model_ang):.0f}° "
      f"(25–75th {np.percentile(model_ang, 25):.0f}–{np.percentile(model_ang, 75):.0f}°)")

H4    at u=-157.4, w= -4.0: interlimb    39° in the trace,    49° in the model (its hinge 0.1 m away)
H7    at u=-126.6, w= -2.2: interlimb    60° in the trace,    33° in the model (its hinge 1.7 m away)
H8    at u=-139.7, w= 28.2: interlimb    60° in the trace,    48° in the model (its hinge 1.0 m away)
H8    at u=-135.4, w= 59.6: interlimb    57° in the trace,   124° in the model (its hinge 2.3 m away)
H13   at u=-105.2, w= -3.9: interlimb    88° in the trace,   110° in the model (its hinge 5.1 m away, at the search edge: left out)
H24   at u=  40.7, w= 31.4: interlimb    39° in the trace,    42° in the model (its hinge 0.3 m away)
H25   at u=  68.2, w= 46.9: interlimb    39° in the trace,    41° in the model (its hinge 4.7 m away, at the search edge: left out)
H25   at u=  97.0, w= 24.8: interlimb    43° in the trace,    53° in the model (its hinge 5.1 m away, at the search edge: left out)
H26   at u=  77.8, w= 28.3: interlimb    47° in the trace,    46° in the model (its hinge 1.3 

## 5. Surfe's interior extrema

Surfe's maxima and minima (05c), and for each: the distance to the nearest datum the fit used (a
trace point or a bedding reading), whether it lies above the platform top, and the distance to the
nearest point of the photographed outcrop (the 05a point cloud).

In [14]:
surfe = np.load(RES_DIR / "surfe.npz")
kinds = surfe["critical_kinds"].astype(str)
ext = surfe["critical_points"][np.isin(kinds, ("maximum", "minimum"))]
to_data = cKDTree(np.vstack([*traces, gp])).query(ext)[0]
to_cloud = cKDTree(data["cloud_xyz"]).query(ext)[0]
above = ext[:, 2] > z_beach
numbers["surfe_extrema"] = int(len(ext))
numbers["surfe_extrema_to_data_m_median"] = float(np.median(to_data))
numbers["surfe_extrema_above_platform"] = int(above.sum())
numbers["surfe_extrema_above_platform_to_outcrop_m_median"] = float(np.median(to_cloud[above])) if above.any() else None
for e, k, dd, dc in zip(ext, kinds[np.isin(kinds, ("maximum", "minimum"))], to_data, to_cloud):
    print(f"{k:8s} z {e[2]:6.1f} m: {dd:5.1f} m from the data, {dc:5.1f} m from the outcrop")
print(f"{len(ext)} extrema: median {np.median(to_data):.1f} m from the data; {above.sum()} above the platform, "
      f"those a median {numbers['surfe_extrema_above_platform_to_outcrop_m_median']:.1f} m from the outcrop")

maximum  z   21.4 m:  11.1 m from the data,  10.9 m from the outcrop
maximum  z   47.7 m:   8.1 m from the data,   7.1 m from the outcrop
maximum  z   80.2 m:  33.3 m from the data,  32.4 m from the outcrop
minimum  z   66.7 m:  10.1 m from the data,   8.5 m from the outcrop
maximum  z   60.0 m:  13.6 m from the data,  10.6 m from the outcrop
maximum  z   56.4 m:  11.1 m from the data,   2.2 m from the outcrop
minimum  z   -9.8 m:   9.8 m from the data,   4.5 m from the outcrop
maximum  z  -19.1 m:   4.2 m from the data,   3.8 m from the outcrop
maximum  z  -18.8 m:   0.7 m from the data,   0.9 m from the outcrop
9 extrema: median 10.1 m from the data; 6 above the platform, those a median 9.6 m from the outcrop


## 6. The ensemble, re-scored from the saved models

05b scores each member as it finishes fitting, but the saved and reloaded member 19 gives a slightly
different restored-depth misfit (above) from 05b's score. Here every saved member is reloaded and scored the
same way as section 1 and 2, so the ensemble range the text quotes comes from the models the figures use.

In [15]:
rows = []
for k in range(len(metrics["seeds"])):
    f = loadModel(RES_DIR / "ensemble" / f"member_{k:02d}.pt").events[0].field
    with torch.no_grad():
        sd = np.median([float(f.inverse_map(T(t))[:, 2].std()) for t in traces])
        pr = f.predict_normals(T(gp)).numpy()
    pr = pr / np.linalg.norm(pr, axis=1, keepdims=True).clip(min=1e-12)
    ang = np.degrees(np.arctan2(np.linalg.norm(np.cross(obs, pr), axis=1), np.clip((obs * pr).sum(1), -1, 1)))
    rows.append((sd, float(np.median(ang)), float(np.median(metrics["depth_std"][k]))))
    print(f"member {k:02d}: restored-depth sd median {sd:.2f} m (05b scored {rows[-1][2]:.2f}), bedding {rows[-1][1]:.1f}°")
sd, ang, sd_05b = (np.array(c) for c in zip(*rows))
numbers["ensemble_depth_sd_m"] = [float(sd.min()), float(sd.max())]
numbers["ensemble_bedding_deg"] = [float(ang.min()), float(ang.max())]
print(f"reloaded: restored-depth sd {sd.min():.2f}–{sd.max():.2f} m, bedding {ang.min():.1f}–{ang.max():.1f}° | "
      f"05b: {sd_05b.min():.2f}–{sd_05b.max():.2f} m")

member 00: restored-depth sd median 0.57 m (05b scored 0.57), bedding 1.6°
member 01: restored-depth sd median 0.59 m (05b scored 0.59), bedding 1.5°
member 02: restored-depth sd median 0.59 m (05b scored 0.59), bedding 1.6°
member 03: restored-depth sd median 0.66 m (05b scored 0.66), bedding 1.7°
member 04: restored-depth sd median 0.53 m (05b scored 0.53), bedding 1.7°
member 05: restored-depth sd median 0.53 m (05b scored 0.53), bedding 1.5°
member 06: restored-depth sd median 0.60 m (05b scored 0.60), bedding 1.9°
member 07: restored-depth sd median 0.60 m (05b scored 0.60), bedding 1.5°
member 08: restored-depth sd median 0.66 m (05b scored 0.66), bedding 1.6°
member 09: restored-depth sd median 0.60 m (05b scored 0.60), bedding 1.6°
member 10: restored-depth sd median 0.63 m (05b scored 0.63), bedding 1.7°
member 11: restored-depth sd median 0.57 m (05b scored 0.57), bedding 1.7°
member 12: restored-depth sd median 0.59 m (05b scored 0.59), bedding 1.6°
member 13: restored-depth

## 7. Every test, for every member

Sections 1–4 and the held-out data, repeated for each of the 20 saved members with the same functions and settings:
bedding misfit (all, cliff, platform), restored-depth misfit, sequence height, shortening from the restored traces (median over the folded beds),
the model's interlimb angles at 05a's hinges (median and interquartile range over the hinges it reproduces), and
the held-out data (the two trace pairs' flatness and the 20 held-out bedding readings). One line per member, then the
median and range over the members.

In [17]:
heldout_names, heldout = hartland_inputs.heldout_traces(data)
gp_test, gv_test = data["gp_test"], data["gv_test"]
obs_test = gv_test / np.linalg.norm(gv_test, axis=1, keepdims=True)


def bedding_angles(f, points, unit_obs):
    with torch.no_grad():
        pr = f.predict_normals(T(points)).numpy()
    pr = pr / np.linalg.norm(pr, axis=1, keepdims=True).clip(min=1e-12)
    return np.degrees(np.arctan2(np.linalg.norm(np.cross(unit_obs, pr), axis=1), np.clip((unit_obs * pr).sum(1), -1, 1)))


def member_tests(k):
    mdl = loadModel(RES_DIR / "ensemble" / f"member_{k:02d}.pt")
    f = mdl.events[0].field

    def sc(xyz):
        with torch.no_grad():
            return np.concatenate([np.asarray(mdl.predict(T(c), litho=False).scalar).ravel()
                                   for c in np.array_split(np.asarray(xyz, float), max(1, len(xyz) // 50_000))])

    def vg(points):
        with torch.no_grad():
            v, g = f.value_and_gradient(T(points))
        return v.numpy().ravel(), g.numpy()

    ang = bedding_angles(f, gp, obs)
    with torch.no_grad():
        rest = [f.inverse_map(T(t)).numpy() for t in traces]
    dep = np.array([r_[:, 2].mean() for r_ in rest])
    sd = np.array([r_[:, 2].std(ddof=1) for r_ in rest])
    Wk = np.array([np.ptp(traces[j] @ u_ax) for j in folded])
    W0k = np.array([np.ptp(rest[j] @ r_ax) for j in folded])
    model_angles = []
    for name, uw in zip(map(str, folds["hinge_trace"]), folds["hinge_uw"]):
        t = all_traces[all_names.index(name)]
        p = t[int(np.argmin(np.linalg.norm(t @ basis - uw, axis=1)))]
        U, Wg = np.meshgrid(uw[0] + uu, uw[1] + uu)
        xyz = (p @ hinge) * hinge + U.ravel()[:, None] * u_ax + Wg.ravel()[:, None] * w_ax
        lines = contourpy.contour_generator(uw[0] + uu, uw[1] + uu, sc(xyz).reshape(U.shape),
                                            line_type="Separate").lines(float(sc(p[None])[0]))
        if not lines:
            continue
        line = min(lines, key=lambda l: np.linalg.norm(l - uw, axis=1).min())
        a, h = interlimb(line, uw, HINGE_SEARCH)
        if np.linalg.norm(h - uw) < HINGE_SEARCH - DS:
            model_angles.append(a)
    model_angles = np.array(model_angles)
    return {
        "bedding": np.median(ang), "bedding_cliff": np.median(ang[~on_beach]), "bedding_platform": np.median(ang[on_beach]),
        "depth_sd": np.median(sd), "height": np.ptp(dep), "shortening": np.median(100 * (1 - Wk / W0k)),
        "interlimb_n": len(model_angles), "interlimb": np.median(model_angles) if len(model_angles) else np.nan,
        "interlimb_q1": np.percentile(model_angles, 25) if len(model_angles) else np.nan,
        "interlimb_q3": np.percentile(model_angles, 75) if len(model_angles) else np.nan,
        "pairs": hartland_inputs.trace_flatness(vg, heldout), "bedding_test": np.median(bedding_angles(f, gp_test, obs_test)),
    }


ens = []
print(f"{'member':>6} | bedding all/cliff/platform (°) | depth sd (m) | height (m) | shortening (%) | "
      f"interlimb median [IQR] (n) | pairs {' / '.join(heldout_names)} (m) | held-out bedding (°)")
for k in range(len(metrics["seeds"])):
    s = member_tests(k)
    ens.append(s)
    print(f"{k:>6} | {s['bedding']:5.1f} / {s['bedding_cliff']:4.1f} / {s['bedding_platform']:4.1f} | {s['depth_sd']:12.2f} | "
          f"{s['height']:10.1f} | {s['shortening']:14.0f} | {s['interlimb']:5.0f} [{s['interlimb_q1']:.0f}–{s['interlimb_q3']:.0f}] ({s['interlimb_n']}) | "
          f"{' / '.join(f'{v:.1f}' for v in s['pairs'])} | {s['bedding_test']:.1f}")


def span(key, fmt="{:.1f}", get=lambda s, key: s[key]):
    v = np.array([get(s, key) for s in ens], float)
    v = v[np.isfinite(v)]
    return f"median {fmt.format(np.median(v))} ({fmt.format(v.min())}–{fmt.format(v.max())})"


print(f"\nover {len(ens)} members, median (min–max):")
print(f"  bedding misfit: all {span('bedding')}°, cliff {span('bedding_cliff')}°, platform {span('bedding_platform')}°")
print(f"  restored-depth sd: {span('depth_sd', '{:.2f}')} m; sequence height: {span('height', '{:.0f}')} m")
print(f"  shortening from the restored traces, median over the beds: {span('shortening', '{:.0f}')}%")
print(f"  interlimb, median over the hinges each member reproduces: {span('interlimb', '{:.0f}')}° "
      f"(hinges reproduced: {span('interlimb_n', '{:.0f}')})")
for j, name in enumerate(heldout_names):
    print(f"  held-out pair {name}: flatness {span(j, get=lambda s, j: s['pairs'][j])} m")
print(f"  held-out bedding: {span('bedding_test')}°")
numbers["ensemble_tests"] = {key: [float(s[key]) for s in ens]
                             for key in ("bedding", "bedding_cliff", "bedding_platform", "depth_sd", "height",
                                         "shortening", "interlimb", "interlimb_n", "bedding_test")}
numbers["ensemble_tests"]["pairs"] = [[float(v) for v in s["pairs"]] for s in ens]

member | bedding all/cliff/platform (°) | depth sd (m) | height (m) | shortening (%) | interlimb median [IQR] (n) | pairs H22+H25 / H1+H7 (m) | held-out bedding (°)
     0 |   1.6 /  2.0 /  1.5 |         0.57 |      105.2 |             38 |    50 [49–53] (8) | 4.9 / 3.8 | 19.1
     1 |   1.5 /  1.8 /  1.5 |         0.59 |      110.6 |             29 |    48 [44–55] (11) | 5.2 / 4.3 | 17.4
     2 |   1.6 /  1.8 /  1.5 |         0.59 |      111.6 |             25 |    51 [44–59] (10) | 8.6 / 4.5 | 16.7
     3 |   1.7 /  1.7 /  1.7 |         0.66 |      108.6 |             36 |    53 [43–63] (10) | 8.5 / 3.9 | 17.3
     4 |   1.7 /  1.7 /  1.7 |         0.53 |      119.1 |             36 |    55 [48–61] (11) | 9.6 / 5.0 | 20.3
     5 |   1.5 /  1.8 /  1.3 |         0.53 |      100.2 |             39 |    51 [43–59] (9) | 7.9 / 5.6 | 14.6
     6 |   1.9 /  2.2 /  1.7 |         0.60 |      118.6 |             25 |    49 [43–60] (8) | 8.1 / 6.7 | 20.7
     7 |   1.5 /  1.6 /  1.4 |         0

## Save

In [18]:
out = RES_DIR / f"numbers_member_{MEMBER:02d}.json"
out.write_text(json.dumps(numbers, indent=1))
print("wrote", out)

wrote results/numbers_member_19.json
